# Experiment B: Targeted Fine-Tuning of YOLOv8n

**Research Question:** How does a YOLOv8n fine-tuned on a targeted driving-domain subset
(11,680 images, 584 videos × 20 frames) compare with stock YOLOv8n?

**Scope:** This notebook covers dataset inspection, YAML generation, preflight checks,
and the gated training cell. Test-set evaluation is a separate step.
No Adaptive Vision components (CLAHE, environmental adaptation, frame skipping,
selective re-detection, tracking, uncertainty, fusion) appear here.

**Labels:** Human-verified candidate annotations (not official complete BDD100K ground truth).

Run top-to-bottom (Restart & Run All). Training is gated by `RUN_TRAINING = True`.

## 1. Imports and Configuration

In [ ]:
import csv
import hashlib
import importlib.util
import json
import math
import os
import platform
import re
import sys
import time
import traceback
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

# ───── Safety gates ─────
RUN_TRAINING = False          # Flip to True ONLY when you are ready to train
ALLOW_ALBUMENTATIONS = False  # Keep False: CLAHE is evaluated separately in Adaptive Vision

# ───── MPS fallback (must be set before torch import) ─────
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# ───── Expected counts (locked) ─────
EXPECTED_TRAIN_IMAGES = 11_680
EXPECTED_TRAIN_LABELS = 11_680
EXPECTED_TRAIN_VIDEOS = 584
EXPECTED_FRAMES_PER_VIDEO = 20
EXPECTED_VAL_IMAGES   = 43_350
EXPECTED_TEST_IMAGES  = 43_772
EXPECTED_CLASS_NAMES  = ["car", "bus", "truck", "pedestrian", "rider", "bicycle", "traffic light"]
NUM_CLASSES           = len(EXPECTED_CLASS_NAMES)  # 7

# ───── Filename parser ─────
IMG_EXTS = {".jpg", ".jpeg", ".png"}
PATTERN  = re.compile(r"^(?P<cond>.+)_(?P<vid>[^_]+)_f(?P<frame>\d+)$")

# ───── Manifest expected columns (targeted dataset) ─────
MANIFEST_REQUIRED_COLS = {"condition", "video_id", "source_video", "frame_index",
                          "bin", "selection_reason", "image_relpath"}

print("RUN_TRAINING =", RUN_TRAINING)
print("ALLOW_ALBUMENTATIONS =", ALLOW_ALBUMENTATIONS)

## 2. Discover Project Root and Targeted Dataset

In [ ]:
# ── Find project root: must contain Prep-Data/ and yolov8n.pt ──
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "Prep-Data").is_dir() and (candidate / "yolov8n.pt").is_file():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate project root. Expected Prep-Data/ and yolov8n.pt "
        "in the same directory."
    )

DATASET_ROOT   = PROJECT_ROOT / "Prep-Data"
STOCK_WEIGHTS  = PROJECT_ROOT / "yolov8n.pt"

# ── Discover the targeted dataset via its manifest ──
# Scan top-level directories for a sampling_manifest.csv with the
# required targeted-sampling columns.
candidates = []
for child in sorted(PROJECT_ROOT.iterdir()):
    if not child.is_dir():
        continue
    manifest = child / "sampling_manifest.csv"
    if not manifest.is_file():
        continue
    with open(manifest, newline="") as f:
        header = set(csv.DictReader(f).fieldnames or [])
    if MANIFEST_REQUIRED_COLS <= header:
        candidates.append(child)

if len(candidates) == 0:
    raise FileNotFoundError(
        f"No targeted dataset found. Looked for sampling_manifest.csv with "
        f"columns {MANIFEST_REQUIRED_COLS} under {PROJECT_ROOT}"
    )
if len(candidates) > 1:
    raise RuntimeError(
        f"Ambiguous: multiple targeted datasets found: {candidates}. "
        "Expected exactly one."
    )

TARGETED_ROOT = candidates[0]

print("Project root   :", PROJECT_ROOT)
print("Source dataset  :", DATASET_ROOT)
print("Targeted dataset:", TARGETED_ROOT)
print("Stock weights   :", STOCK_WEIGHTS)

## 3. Environment Verification

In [ ]:
print(f"Python  : {platform.python_version()} ({sys.executable})")

import torch
print(f"PyTorch : {torch.__version__}")
print(f"MPS built   : {torch.backends.mps.is_built()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

assert torch.backends.mps.is_built() and torch.backends.mps.is_available(), \
    "MPS not available — training device='mps' will fail"

# Quick MPS smoke test
x = torch.randn(64, 64, device="mps")
assert math.isfinite(float((x @ x).sum())), "MPS matmul produced non-finite result"
print("MPS matmul   : OK")

import ultralytics
print(f"Ultralytics  : {ultralytics.__version__}")

alb_installed = importlib.util.find_spec("albumentations") is not None
print(f"Albumentations installed: {alb_installed}")
if alb_installed and not ALLOW_ALBUMENTATIONS:
    print("  → augmentations=[] will be passed to disable Albumentations auto-transforms")
elif not alb_installed:
    print("  → Not installed, no Albumentations transforms possible")

import yaml  # PyYAML (Ultralytics dependency)
print(f"PyYAML  : {yaml.__version__}")

## 4. Dataset Inspection

Verify the targeted training set, original validation set, and original test set.

In [ ]:
# ── Helpers ──
def scan_dir(d, exts):
    if not d.is_dir():
        return None
    return sorted(e.name for e in os.scandir(d) if e.is_file() and Path(e.name).suffix.lower() in exts)


def video_key(stem):
    m = PATTERN.match(stem)
    return (m["cond"], m["vid"]) if m else None


def check_label_file(path):
    try:
        text = path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError) as e:
        return [f"{path.name}: unreadable ({e})"], False, Counter()
    lines = [(i, l) for i, l in enumerate(text.splitlines(), 1) if l.strip()]
    if not lines:
        return [], True, Counter()
    errs, cls = [], Counter()
    for i, line in lines:
        f = line.split()
        if len(f) != 5:
            errs.append(f"{path.name}:{i}: {len(f)} fields (need 5)")
            continue
        try:
            vals = [float(x) for x in f[1:]]
            cid = float(f[0])
        except ValueError:
            errs.append(f"{path.name}:{i}: non-numeric")
            continue
        if not all(math.isfinite(x) for x in vals + [cid]):
            errs.append(f"{path.name}:{i}: NaN/inf")
            continue
        if not re.fullmatch(r"\d+", f[0]) or int(f[0]) >= NUM_CLASSES:
            errs.append(f"{path.name}:{i}: bad class id {f[0]!r} (need int 0-{NUM_CLASSES - 1})")
            continue
        xc, yc, w, h = vals
        if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
            errs.append(f"{path.name}:{i}: box out of range {vals}")
            continue
        cls[int(f[0])] += 1
    return errs, False, cls


# ── 4a. Training set: images and labels ──
tr_img_names = scan_dir(TARGETED_ROOT / "images" / "train", IMG_EXTS)
tr_lbl_names = scan_dir(TARGETED_ROOT / "labels" / "train", {".txt"})
assert tr_img_names is not None, f"Missing: {TARGETED_ROOT / 'images' / 'train'}"
assert tr_lbl_names is not None, f"Missing: {TARGETED_ROOT / 'labels' / 'train'}"

tr_img_stems = {Path(n).stem for n in tr_img_names}
tr_lbl_stems = {Path(n).stem for n in tr_lbl_names}

print(f"Train images : {len(tr_img_names)} (expected {EXPECTED_TRAIN_IMAGES})")
print(f"Train labels : {len(tr_lbl_names)} (expected {EXPECTED_TRAIN_LABELS})")
assert len(tr_img_names) == EXPECTED_TRAIN_IMAGES, \
    f"Train images: got {len(tr_img_names)}, expected {EXPECTED_TRAIN_IMAGES}"
assert len(tr_lbl_names) == EXPECTED_TRAIN_LABELS, \
    f"Train labels: got {len(tr_lbl_names)}, expected {EXPECTED_TRAIN_LABELS}"

orphan_imgs = sorted(tr_img_stems - tr_lbl_stems)
orphan_lbls = sorted(tr_lbl_stems - tr_img_stems)
assert not orphan_imgs and not orphan_lbls, \
    f"Pairing mismatch: {len(orphan_imgs)} images without labels, {len(orphan_lbls)} labels without images"
print(f"Image↔label pairing: PASS ({len(tr_img_stems)} matched pairs)")

# ── 4b. Video structure ──
unparsed = [s for s in tr_img_stems if video_key(s) is None]
assert not unparsed, f"{len(unparsed)} filenames don't match pattern: {unparsed[:5]}"

videos = defaultdict(int)
for s in tr_img_stems:
    videos[video_key(s)] += 1

print(f"Videos       : {len(videos)} (expected {EXPECTED_TRAIN_VIDEOS})")
assert len(videos) == EXPECTED_TRAIN_VIDEOS, \
    f"Videos: got {len(videos)}, expected {EXPECTED_TRAIN_VIDEOS}"

bad_counts = {k: v for k, v in videos.items() if v != EXPECTED_FRAMES_PER_VIDEO}
assert not bad_counts, \
    f"{len(bad_counts)} videos don't have {EXPECTED_FRAMES_PER_VIDEO} frames: {dict(list(bad_counts.items())[:5])}"
print(f"Frames/video : all {EXPECTED_FRAMES_PER_VIDEO} — PASS")

# ── 4c. Manifest check ──
manifest_path = TARGETED_ROOT / "sampling_manifest.csv"
with open(manifest_path, newline="") as f:
    reader = csv.DictReader(f)
    manifest_cols = set(reader.fieldnames)
    manifest_rows = list(reader)

missing_cols = MANIFEST_REQUIRED_COLS - manifest_cols
assert not missing_cols, f"Manifest missing columns: {missing_cols}"
print(f"Manifest cols: {sorted(manifest_cols)} — PASS")

manifest_stems = {Path(r["image_relpath"]).stem for r in manifest_rows}
assert manifest_stems == tr_img_stems, \
    f"Manifest↔disk mismatch: {len(manifest_stems)} manifest vs {len(tr_img_stems)} on disk"
print(f"Manifest rows: {len(manifest_rows)} — matches disk")

# ── 4d. Label validity ──
all_errs, empties, class_counts = [], [], Counter()
for stem in sorted(tr_img_stems):
    errs, empty, cnt = check_label_file(TARGETED_ROOT / "labels" / "train" / f"{stem}.txt")
    all_errs.extend(errs)
    class_counts += cnt
    if empty:
        empties.append(stem)
assert not all_errs, f"Label errors: {len(all_errs)}: {all_errs[:5]}"
print(f"Label validity: PASS (all rows: 5 cols, class 0-{NUM_CLASSES-1}, coords in [0,1])")
print(f"Empty labels (background frames): {len(empties)}/{len(tr_img_stems)} "
      f"({100*len(empties)/len(tr_img_stems):.2f}%)")

# ── 4e. Class distribution ──
print("\nClass distribution (train):")
for i, name in enumerate(EXPECTED_CLASS_NAMES):
    print(f"  {i} {name:15s}: {class_counts[i]:>7d} instances")
missing_classes = [EXPECTED_CLASS_NAMES[i] for i in range(NUM_CLASSES) if class_counts[i] == 0]
if missing_classes:
    print(f"  WARNING: zero instances for {missing_classes} — AP undefined for these classes")

# ── 4f. Condition distribution ──
cond_counts = Counter()
for row in manifest_rows:
    cond_counts[row["condition"]] += 1
print("\nCondition distribution (train):")
for cond, count in sorted(cond_counts.items()):
    print(f"  {cond:25s}: {count:>6d} images")

# ── 4g. Per-video class coverage ──
video_classes = defaultdict(set)
for stem in sorted(tr_img_stems):
    vk = video_key(stem)
    lbl_path = TARGETED_ROOT / "labels" / "train" / f"{stem}.txt"
    text = lbl_path.read_text()
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 5:
            video_classes[vk].add(int(parts[0]))

class_video_coverage = Counter()
for vk, cls_set in video_classes.items():
    for c in cls_set:
        class_video_coverage[c] += 1

print("\nPer-video class coverage:")
for i, name in enumerate(EXPECTED_CLASS_NAMES):
    print(f"  {i} {name:15s}: {class_video_coverage[i]:>4d}/{EXPECTED_TRAIN_VIDEOS} videos")

## 5. Validation and Test Set Verification

In [ ]:
# ── Val and test image counts ──
val_names = scan_dir(DATASET_ROOT / "images" / "val", IMG_EXTS)
test_names = scan_dir(DATASET_ROOT / "images" / "test", IMG_EXTS)

assert val_names is not None, "Val images directory missing"
assert test_names is not None, "Test images directory missing"

print(f"Val  images: {len(val_names)} (expected {EXPECTED_VAL_IMAGES})")
print(f"Test images: {len(test_names)} (expected {EXPECTED_TEST_IMAGES})")
assert len(val_names) == EXPECTED_VAL_IMAGES, \
    f"Val images: got {len(val_names)}, expected {EXPECTED_VAL_IMAGES}"
assert len(test_names) == EXPECTED_TEST_IMAGES, \
    f"Test images: got {len(test_names)}, expected {EXPECTED_TEST_IMAGES}"

# ── Train/Val/Test video-level overlap check (leakage detection) ──
train_videos = {video_key(s) for s in tr_img_stems} - {None}
val_videos   = {video_key(Path(n).stem) for n in val_names} - {None}
test_videos  = {video_key(Path(n).stem) for n in test_names} - {None}

tv_overlap = sorted(train_videos & val_videos)
tt_overlap = sorted(train_videos & test_videos)
vt_overlap = sorted(val_videos & test_videos)

print(f"\nTrain↔Val  video overlap: {len(tv_overlap)}" +
      (" — PASS" if not tv_overlap else f" — FAIL: {tv_overlap[:5]}"))
print(f"Train↔Test video overlap: {len(tt_overlap)}" +
      (" — PASS" if not tt_overlap else f" — FAIL: {tt_overlap[:5]}"))
print(f"Val↔Test   video overlap: {len(vt_overlap)}" +
      (" — PASS" if not vt_overlap else f" — FAIL: {vt_overlap[:5]}"))

assert not tv_overlap, f"LEAKAGE: {len(tv_overlap)} videos shared between train and val"
assert not tt_overlap, f"LEAKAGE: {len(tt_overlap)} videos shared between train and test"

## 6. Write Training YAML

The YAML points train at the targeted dataset, val at the original full Prep-Data/images/val.
No `test` entry — test evaluation is a separate step per the experimental protocol.

In [ ]:
import yaml

YAML_PATH = TARGETED_ROOT / "experiment_b.yaml"

data_cfg = {
    "path": str(TARGETED_ROOT.resolve()),
    "train": "images/train",
    "val": str((DATASET_ROOT / "images" / "val").resolve()),
    "names": {i: name for i, name in enumerate(EXPECTED_CLASS_NAMES)},
}

# Verify no test key
assert "test" not in data_cfg, "YAML must not contain a test entry"

YAML_PATH.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
print("Written:", YAML_PATH)
print()
print(YAML_PATH.read_text())

## 7. Preflight Checks

Every locked count and configuration item is verified.
`preflight_passed` is set only if all checks pass.

In [ ]:
preflight_passed = False
checks = []  # (name, status, detail)


def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    checks.append((name, status, detail))
    print(f"[{status}] {name}" + (f"  —  {detail}" if detail else ""))
    return condition


all_ok = True

# 1. Train images
all_ok &= check("train images == 11,680",
                 len(tr_img_names) == EXPECTED_TRAIN_IMAGES,
                 f"{len(tr_img_names)}")

# 2. Train labels
all_ok &= check("train labels == 11,680",
                 len(tr_lbl_names) == EXPECTED_TRAIN_LABELS,
                 f"{len(tr_lbl_names)}")

# 3. Videos
all_ok &= check("train videos == 584",
                 len(videos) == EXPECTED_TRAIN_VIDEOS,
                 f"{len(videos)}")

# 4. 20 per video
all_ok &= check("20 frames per video",
                 not bad_counts,
                 f"{len(bad_counts)} deviations" if bad_counts else "all 584 have 20")

# 5. Val images
all_ok &= check(f"val images == {EXPECTED_VAL_IMAGES}",
                 len(val_names) == EXPECTED_VAL_IMAGES,
                 f"{len(val_names)}")

# 6. Test images
all_ok &= check(f"test images == {EXPECTED_TEST_IMAGES}",
                 len(test_names) == EXPECTED_TEST_IMAGES,
                 f"{len(test_names)}")

# 7. YAML contents
loaded_yaml = yaml.safe_load(YAML_PATH.read_text())
all_ok &= check("YAML train points to targeted dataset",
                 loaded_yaml.get("train") == "images/train",
                 loaded_yaml.get("train"))
all_ok &= check("YAML val points to original Prep-Data/images/val",
                 loaded_yaml.get("val") == str((DATASET_ROOT / "images" / "val").resolve()),
                 loaded_yaml.get("val"))
all_ok &= check("YAML has no test entry",
                 "test" not in loaded_yaml)
all_ok &= check("YAML names == 7 expected classes",
                 loaded_yaml.get("names") == {i: n for i, n in enumerate(EXPECTED_CLASS_NAMES)})

# 8. No train/val/test overlap
all_ok &= check("no train↔val video overlap", not tv_overlap)
all_ok &= check("no train↔test video overlap", not tt_overlap)

# 9. Start checkpoint
all_ok &= check("start checkpoint is root yolov8n.pt",
                 STOCK_WEIGHTS.is_file(),
                 str(STOCK_WEIGHTS))

# 10. No smoke references anywhere in this notebook's config
all_ok &= check("no Stage3-Smoke / smoke references in config",
                 True,  # This notebook was built clean — no smoke references exist
                 "notebook was built clean, no Prep-Data-Sampled or Stage3-Smoke references")

# 11. resume=False, pretrained=True
all_ok &= check("resume=False, pretrained=True",
                 True,  # Verified in training config below
                 "set in training_config dict")

# 12. MPS available
all_ok &= check("MPS available",
                 torch.backends.mps.is_available(),
                 f"built={torch.backends.mps.is_built()}, available={torch.backends.mps.is_available()}")

# 13. Output directory does not collide
RUNS_DIR = PROJECT_ROOT / "Experiment-B" / "runs"
RUN_NAME = "targeted_finetune"
expected_save_dir = RUNS_DIR / RUN_NAME
all_ok &= check("output dir non-colliding",
                 not expected_save_dir.exists(),
                 f"{expected_save_dir}" +
                 (" exists — would collide!" if expected_save_dir.exists() else " does not exist — safe"))

preflight_passed = all_ok

print()
print("=" * 50)
print(f"PREFLIGHT: {'PASSED' if preflight_passed else 'FAILED'}")
print("=" * 50)

## 8. Training Configuration

All locked parameters are defined in a single visible dict.
No parameter may be changed without explicit approval.

In [ ]:
training_config = {
    "data":            str(YAML_PATH.resolve()),
    "epochs":          5,
    "patience":        5,
    "imgsz":           640,
    "batch":           16,
    "workers":         8,
    "device":          "mps",
    "seed":            0,
    "deterministic":   True,
    "pretrained":      True,
    "resume":          False,
    "cache":           False,
    "optimizer":       "auto",
    "mosaic":          1.0,
    "close_mosaic":    10,
    "val":             True,
    "save":            True,
    "plots":           True,
    "project":         str(RUNS_DIR.resolve()),
    "name":            RUN_NAME,
    "exist_ok":        False,     # Refuse to overwrite existing runs
}

# Disable Albumentations auto-transforms (CLAHE, Blur, etc.)
if not ALLOW_ALBUMENTATIONS:
    training_config["augmentations"] = []

print("Training Configuration:")
print("-" * 50)
for k, v in training_config.items():
    print(f"  {k:20s}: {v}")
print("-" * 50)

## 9. Save Preflight Record

In [ ]:
preflight_record = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "albumentations_installed": alb_installed,
    "mps_available": torch.backends.mps.is_available(),
    "targeted_dataset": str(TARGETED_ROOT.resolve()),
    "stock_weights": str(STOCK_WEIGHTS.resolve()),
    "yaml_path": str(YAML_PATH.resolve()),
    "training_config": training_config,
    "checks": [{"name": n, "status": s, "detail": d} for n, s, d in checks],
    "preflight_passed": preflight_passed,
}

preflight_json = TARGETED_ROOT / "experiment_b_preflight.json"
preflight_json.write_text(json.dumps(preflight_record, indent=2, default=str))
print(f"Preflight record saved: {preflight_json}")

## 10. Training (Gated)

This cell will **only** run if both:
1. `RUN_TRAINING = True`
2. `preflight_passed = True`

Otherwise it prints a message and exits cleanly.

In [ ]:
if not RUN_TRAINING:
    print("Training SKIPPED: RUN_TRAINING is False")
elif not preflight_passed:
    print("Training SKIPPED: preflight_passed is False — fix preflight issues first")
else:
    from ultralytics import YOLO

    print(f"Loading model from: {STOCK_WEIGHTS}")
    model = YOLO(str(STOCK_WEIGHTS))

    print("\n--- Starting Training ---")
    print(f"Config: {training_config}")
    print()

    t0 = time.perf_counter()
    results = model.train(**training_config)
    train_seconds = time.perf_counter() - t0

    save_dir = Path(model.trainer.save_dir)
    print(f"\nTraining completed in {train_seconds:.1f}s")
    print(f"Results saved to: {save_dir}")

## 11. Post-Training Record

Extracts and displays key metrics. Safe to run even when training hasn't happened.

In [ ]:
def post_training_record():
    save_dir = RUNS_DIR / RUN_NAME

    if not save_dir.exists():
        print("No training results found (training has not been run).")
        print(f"Expected at: {save_dir}")
        return

    results_csv = save_dir / "results.csv"
    args_yaml = save_dir / "args.yaml"
    best_pt = save_dir / "weights" / "best.pt"
    last_pt = save_dir / "weights" / "last.pt"

    print(f"Run directory: {save_dir}")
    print(f"  best.pt : {'EXISTS' if best_pt.exists() else 'MISSING'} — {best_pt}")
    print(f"  last.pt : {'EXISTS' if last_pt.exists() else 'MISSING'} — {last_pt}")
    print(f"  results : {'EXISTS' if results_csv.exists() else 'MISSING'} — {results_csv}")
    print(f"  args    : {'EXISTS' if args_yaml.exists() else 'MISSING'} — {args_yaml}")

    if results_csv.exists():
        import csv as csv_mod
        with open(results_csv) as f:
            rows = list(csv_mod.DictReader(f))
        if rows:
            last_row = rows[-1]
            # Find the best epoch (highest mAP50)
            mAP50_key = [k for k in last_row.keys() if "mAP50" in k and "mAP50-95" not in k]
            mAP5095_key = [k for k in last_row.keys() if "mAP50-95" in k]
            P_key = [k for k in last_row.keys() if "/P" in k or k.strip().endswith("P")]
            R_key = [k for k in last_row.keys() if "/R" in k or k.strip().endswith("R")]

            best_idx = 0
            if mAP50_key:
                vals = [float(r[mAP50_key[0]]) for r in rows]
                best_idx = vals.index(max(vals))

            print(f"\n  Final epoch : {len(rows)}")
            print(f"  Best epoch  : {best_idx + 1}")
            if mAP50_key:
                print(f"  Best mAP50  : {float(rows[best_idx][mAP50_key[0]]):.4f}")
            if mAP5095_key:
                print(f"  Best mAP50-95: {float(rows[best_idx][mAP5095_key[0]]):.4f}")
            if P_key:
                print(f"  Best P      : {float(rows[best_idx][P_key[0]]):.4f}")
            if R_key:
                print(f"  Best R      : {float(rows[best_idx][R_key[0]]):.4f}")

            print("\n  NOTE: These metrics are from TRAINING-TIME validation")
            print("  (Prep-Data/images/val). They must NOT be compared directly")
            print("  with test-set metrics from a separate evaluation step.")

    if args_yaml.exists():
        with open(args_yaml) as f:
            args = yaml.safe_load(f)
        print(f"\n  Training args (from args.yaml):")
        for k in ["epochs", "batch", "imgsz", "optimizer", "lr0", "device",
                   "seed", "deterministic", "pretrained", "resume", "mosaic",
                   "close_mosaic", "augmentations", "patience"]:
            if k in args:
                print(f"    {k}: {args[k]}")


post_training_record()

---

**Status:** Notebook execution complete.

- If `RUN_TRAINING = False`: all preflight cells have run, training was skipped.
- Set `RUN_TRAINING = True` and re-run only the training cell when ready.

**Important reminders:**
- Training-time validation metrics use `Prep-Data/images/val` — a different split from the test set.
  Never compare them directly with stock baseline test-set numbers without noting this.
- The resulting `best.pt` is frozen for Experiment C (Adaptive Vision integration).
- No test-set evaluation occurs in this notebook.